Installer Spark

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

Définir les variables d'environnement

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

Importer les data


In [ ]:
!wget wget https://datasets.imdbws.com/title.basics.tsv.gz


--2024-11-04 21:20:11--  http://wget/
Resolving wget (wget)... failed: Name or service not known.
wget: unable to resolve host address ‘wget’
--2024-11-04 21:20:11--  https://datasets.imdbws.com/title.basics.tsv.gz
Resolving datasets.imdbws.com (datasets.imdbws.com)... 3.165.160.46, 3.165.160.37, 3.165.160.80, ...
Connecting to datasets.imdbws.com (datasets.imdbws.com)|3.165.160.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 196919967 (188M) [binary/octet-stream]
Saving to: ‘title.basics.tsv.gz’

title.basics.tsv.gz 100%[===================>] 187.80M   178MB/s    in 1.1s    

2024-11-04 21:20:12 (178 MB/s) - ‘title.basics.tsv.gz’ saved [196919967/196919967]

FINISHED --2024-11-04 21:20:12--
Total wall clock time: 1.2s
Downloaded: 1 files, 188M in 1.1s (178 MB/s)


In [ ]:
import gzip,shutil


In [ ]:
with gzip.open('title.basics.tsv.gz', 'rb') as f_in:
    with open('title.basics.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

Parcourir les data

In [ ]:
# Load data from csv to a dataframe.
# header=True means the first row is a header
# sep=';' means the column are seperated using ''
df_title = spark.read.csv('./title.basics.tsv', header=True, sep="\t")
df_title.show(5)

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0000001|    short|          Carmencita|          Carmencita|      0|     1894|     \N|             1|   Documentary,Short|
|tt0000002|    short|Le clown et ses c...|Le clown et ses c...|      0|     1892|     \N|             5|     Animation,Short|
|tt0000003|    short|        Poor Pierrot|      Pauvre Pierrot|      0|     1892|     \N|             5|Animation,Comedy,...|
|tt0000004|    short|         Un bon bock|         Un bon bock|      0|     1892|     \N|            12|     Animation,Short|
|tt0000005|    short|    Blacksmith Scene|    Blacksmith Scene|      0|     1893|     \N|             1|        Comedy

Filtrer

In [ ]:
# Register Temporary Table
df_title.createOrReplaceTempView("temp")
# Select all data from temp table
spark.sql("select * from temp where genres like '%Short' and startYear > 2000 limit 5 ").show(truncate=False)

+---------+---------+--------------------------------+--------------------------------+-------+---------+-------+--------------+------------+
|tconst   |titleType|primaryTitle                    |originalTitle                   |isAdult|startYear|endYear|runtimeMinutes|genres      |
+---------+---------+--------------------------------+--------------------------------+-------+---------+-------+--------------+------------+
|tt0050396|short    |Final Curtain                   |Final Curtain                   |0      |2012     |\N     |22            |Horror,Short|
|tt0052146|short    |Rondo                           |Rondo                           |0      |2007     |\N     |15            |Short       |
|tt0056840|short    |Aufsätze                        |Aufsätze                        |0      |2021     |\N     |10            |Short       |
|tt0057369|short    |Number 14: Late Superimpositions|Number 14: Late Superimpositions|0      |2023     |\N     |30            |Short       |
|tt006